# Parent-Child Generators

## Define 2 generators 

In [190]:
def generator_1():
    yield 1
    yield 2
    yield 3

def generator_2():
    yield 'a'
    yield 'b'
    yield 'c'
    yield 'd'

## Define a 3rd generator that delegates to the other two

In [191]:
def generator_3():
    yield from generator_1()
    yield from generator_2()

In [192]:
for item in generator_3():
    print(item)

1
2
3
a
b
c
d


Notes:
1. The `yield from` delegates to the child generator
2. The consumer doesn't realize there are two generators behind the scenes

### Capture Return Value from child generators

In [193]:
def generator_1():
    yield 1
    yield 2
    yield 3
    return 'generator_1 exhausted'

def generator_2():
    yield 'a'
    yield 'b'
    yield 'c'
    yield 'd'
    return 'generator_2 exhausted'

In [194]:
def generator_3():
    result1 = yield from generator_1()
    result2 = yield from generator_2()
    yield f'Both {result1} and {result2}'

In [195]:
for item in generator_3():
    print(item)

1
2
3
a
b
c
d
Both generator_1 exhausted and generator_2 exhausted


Notes:
1. `yield from` allows capturing return values from children in parent
2. parent doesn't have to handle a StopIteration

## A more realistic example

### Ops queue for Account Opening Requests

In [149]:
def account_opening_requests():
    clients = ['John', 'Jane', 'Jill']
    status = []
    for client in clients:
        approval_status = yield f'{client} would like to Open an Account. Do you Approve/Reject ?'
        status.append(approval_status)
    return dict(zip(clients, status))

### Ops queue for Loan Requests

In [150]:
def loan_requests():
    clients = ['Robin', 'Roderick', 'Robert']
    status = []
    for client in clients:
        approval_status = yield f'{client} is applying for a loan. Do you Approve / Reject ?'
        status.append(approval_status)
    return dict(zip(clients, status))

### Queue Facade delegates to either queue

In [171]:
def queue_facade():

    num_items = 0

    yield "Welcome to Operations Corner !!! How are you today ?"
    answer = yield "Would you like to process Account Opening Requests ? Say Yes/No"
    if answer.lower() == 'yes':
        result = yield from account_opening_requests()
        num_items += len(result)
        yield f'No more requests. Here is what you processed: {result}. Looks OK ?'
        
    answer = yield "Would you like to process Loan Requests Requests ? Say Yes/No"
    if answer.lower() == 'yes':
        result = yield from loan_requests()
        num_items += len(result)
        yield f'No more requests. Here is what you processed: {result}. Looks OK ?'
        
    yield f'You are all done. You processed {num_items} items today. Thank you and have a good day'

In [172]:
chatbot = queue_facade()

In [173]:
chatbot.send(None)

'Welcome to Operations Corner !!! How are you today ?'

In [174]:
chatbot.send('fine')

'Would you like to process Account Opening Requests ? Say Yes/No'

In [175]:
chatbot.send('yes')

'John would like to Open an Account. Do you Approve/Reject ?'

In [176]:
chatbot.send('approve')

'Jane would like to Open an Account. Do you Approve/Reject ?'

In [177]:
chatbot.send('reject')

'Jill would like to Open an Account. Do you Approve/Reject ?'

In [178]:
chatbot.send('approve')

"No more requests. Here is what you processed: {'John': 'approve', 'Jane': 'reject', 'Jill': 'approve'}. Looks OK ?"

In [179]:
chatbot.send('looks good to me')

'Would you like to process Loan Requests Requests ? Say Yes/No'

In [180]:
chatbot.send('yes')

'Robin is applying for a loan. Do you Approve / Reject ?'

In [181]:
chatbot.send('approve')

'Roderick is applying for a loan. Do you Approve / Reject ?'

In [182]:
chatbot.send('approve')

'Robert is applying for a loan. Do you Approve / Reject ?'

In [183]:
chatbot.send('reject')

"No more requests. Here is what you processed: {'Robin': 'approve', 'Roderick': 'approve', 'Robert': 'reject'}. Looks OK ?"

In [184]:
chatbot.send('looks fine to me')

'You are all done. You processed 6 items today. Thank you and have a good day'

In [189]:
# chatbot.send('thanks')

### Create a chatbot experience

In [188]:
chatbot = queue_facade()
user_message = None
try:
    while True:
        assistant_message = chatbot.send(user_message)
        print('\nAssistant: ', assistant_message, '\n')
        user_message = input('You: ')
except StopIteration as e:
    pass


Assistant:  Welcome to Operations Corner !!! How are you today ? 



You:  fine



Assistant:  Would you like to process Account Opening Requests ? Say Yes/No 



You:  yes



Assistant:  John would like to Open an Account. Do you Approve/Reject ? 



...

You:  reject



Assistant:  Would you like to process Loan Requests Requests ? Say Yes/No 



You:  approve



Assistant:  You are all done. You processed 3 items today. Thank you and have a good day 



You:  approve
